In [12]:
import pandas as pd 
import numpy as np
import os
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB 
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder,StandardScaler



In [2]:
to_file = r'D:\6654305\Du_An_Quan_Trading\Booting\features'
files = [f for f in os.listdir(to_file) if f.endswith('.csv')]
dfs = [pd.read_csv(os.path.join(to_file, file)) for file in files]
df= pd.concat(dfs, ignore_index=True)

In [3]:
df['TradingDate'] = pd.to_datetime(df['TradingDate'], format='%d/%m/%Y', errors='coerce')
df = df.drop(columns="Time")
df= df.dropna()
df.isnull().sum()

Symbol         0
Market         0
TradingDate    0
Open           0
High           0
Low            0
Close          0
Volume         0
Value          0
SMA_20         0
EMA_20         0
RSI            0
MACD           0
Signal_Line    0
Middle_Band    0
Upper_Band     0
Lower_Band     0
TR             0
ATR            0
OBV            0
VMA_20         0
%K             0
%D             0
ROC            0
MFI            0
Williams_%R    0
dtype: int64

In [4]:
cols = df[['SMA_20', 'EMA_20', 'RSI', 'MACD', 'Signal_Line',
       'Middle_Band', 'Upper_Band', 'Lower_Band', 'ATR', 'OBV', 'VMA_20',
       '%K', '%D', 'ROC', 'MFI', 'Williams_%R']]

In [5]:
LB = LabelEncoder()
df["Market"] = LB.fit_transform(df["Market"])
df["Symbol"]= LB.fit_transform(df["Symbol"])

In [14]:
df["Open_T+1"] = df["Open"].shift(-1)
df["Close_T+3"] = df["Close"].shift(-3)
df["Label"] = (df["Close_T+3"] > 1.02*df["Open_T+1"]).astype(float)
data= df.drop(columns=["Open_T+1","Close_T+3"])

In [15]:
data["Day"] = data["TradingDate"].dt.day
data["Month"]  = data["TradingDate"].dt.month
data["Year"]  =data["TradingDate"].dt.year

In [13]:
#Ma hoa cos_sin
data["Day_sin"] = np.sin(2*np.pi*data["Day"]/365)
data["Day_cos"] = np.cos(2*np.pi*data["Day"]/365)

data["Month_sin"] = np.sin(2*np.pi*data["Month"]/12)
data["Month_cos"] = np.cos(2*np.pi*data["Month"]/12)

#data["Year_sin"] = np.sin(2*np.pi*data["Year"]/365)
#data["Year_cos"] = np.cos(2*np.pi*data["Year"]/365)

In [16]:
#Ti le giua cac gia tri
data["Hight/Close"] = data["High"]/data["Close"]
data["Low/Close"] = data["Low"]/data["Close"]
data["Range_PCT"] = (data["High"]-data["Low"])/data["Close"]


In [23]:
#Dac trung ve dong tien
data["Money_flow"] = data["Close"]*data["Volume"]
data["Cumsum_Money_flow"] = data["Money_flow"].cumsum()


In [24]:
data = data.replace([float("inf"), -float("inf")], float("nan")).dropna()


In [25]:
df_train = data[data["TradingDate"] < '2024-01-01']
df_test = data[data["TradingDate"] >= '2024-01-01']

In [26]:
X_train,y_train = df_train.drop(columns=["TradingDate","Label"]),df_train["Label"]
X_test,y_test = df_test.drop(columns=["TradingDate","Label"]),df_test["Label"]

In [20]:
def Model_1(X_train,y_train,x_test,y_test):
    
    model_tree = DecisionTreeClassifier()
    model_tree.fit(X_train,y_train)
    y_pred_tree = model_tree.predict_proba(X_test)[:,1]
    print("Do chinh xac Model tree: ")
    print(roc_auc_score(y_test,y_pred_tree))

    model_bayes = GaussianNB()
    model_bayes.fit(X_train,y_train)
    y_pred_bayes = model_bayes.predict_proba(X_test)[:,1]
    print("Do chinh xac Model bayes: ")
    print(roc_auc_score(y_test,y_pred_bayes))

    model_knn = KNeighborsClassifier(n_neighbors=10)
    model_knn.fit(X_train,y_train)
    y_pred_knn = model_knn.predict_proba(X_test)[:,1]
    print("Do chinh xac Model knn: ")
    print(roc_auc_score(y_test,y_pred_knn))


In [21]:
def Model_2(X_train,y_train,x_test,y_test):
    
    model_MLP = MLPClassifier(hidden_layer_sizes=(30,30),max_iter=200)
    model_MLP.fit(X_train,y_train)
    y_pred_MLP = model_MLP.predict(X_test)
    print("Do chinh xac Model MLP: ")
    print(classification_report(y_pred_MLP,y_test))

    model_RFC = RandomForestClassifier(n_estimators=20)
    model_RFC.fit(X_train,y_train)
    y_pred_RFC = model_RFC.predict(X_test)
    print("Do chinh xac Model RFC: ")
    print(classification_report(y_pred_RFC,y_test))

In [27]:
Model_1(X_train,y_train,X_test,y_test)

Do chinh xac Model tree: 
0.5016819344102358
Do chinh xac Model bayes: 
0.5207728192328553
Do chinh xac Model knn: 
0.499141584208701


In [15]:
Model_2(X_train,y_train,X_test,y_test)

Do chinh xac Model MLP: 
              precision    recall  f1-score   support

         0.0       1.00      0.80      0.89     51120
         1.0       0.00      0.00      0.00         0

    accuracy                           0.80     51120
   macro avg       0.50      0.40      0.44     51120
weighted avg       1.00      0.80      0.89     51120



c:\Users\lenovo\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\lenovo\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\lenovo\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, le

Do chinh xac Model RFC: 
              precision    recall  f1-score   support

         0.0       0.96      0.80      0.87     48980
         1.0       0.05      0.24      0.08      2140

    accuracy                           0.78     51120
   macro avg       0.50      0.52      0.48     51120
weighted avg       0.92      0.78      0.84     51120



In [16]:
def Model(X,y):
    #Chia thanh tap train,test de huan luyen
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3)
    #Model Tree
    model_tree = DecisionTreeClassifier()
    model_tree.fit(X_train,y_train)
    y_pred_tree = model_tree.predict(X_test)
    print("Do chinh xac Model tree: ")
    print(classification_report(y_test,y_pred_tree))
    #Model RFC 
    model_RFC = RandomForestClassifier(n_estimators=20)
    model_RFC.fit(X_train,y_train)
    y_pred_RFC = model_RFC.predict(X_test)
    print("Do chinh xac Model RFC: ")
    print(classification_report(y_test,y_pred_RFC))
